## Use if you have an AOI

In [ ]:
# =========================
# CELL 1 (UPDATED + DEBUGGED):
# Find + print best FIXED lags (L8+L9) using server-side CC_OCEAN ranking
# FIX: REFERENCE_INDEX is a short key, so resolve target image by search
# instead of ee.Image(f"{collection}/{REFERENCE_INDEX}")
# =========================

import ee
from datetime import datetime
from pathlib import Path
import re

from ee_ipl_uv import multitemporal_cloud_masking_2

ee.Initialize()

# -------------------------
# USER SETTINGS
# -------------------------
SCENES_DIR = Path(r"Y:/Mingyue/Timor_part1/landsat_c2_l2_extracted")
MAX_SCENES_TO_PROCESS = None

POLY = [
    [-83.70, 25.12],
    [-83.03, 25.12],
    [-83.03, 26.06],
    [-83.70, 26.06],
    [-83.70, 25.12],
]

region_of_interest = ee.Geometry.Polygon(POLY)

# Lag selection knobs
NUM_LAGS = 5
LAG_WINDOW_YEARS = 7
CC_OCEAN_MAX_PCT = 1.0
EXCLUDE_DAYS_AROUND_TARGET = 7

# Ranking / printing
TOP_N_PRINT = 400
CC_SCALE = 120
TILESCALE = 4
CLOUD_DILATE_M = 60

# NEW: resolver search padding
SEARCH_PAD_DAYS = 2

# -------------------------
# Helpers
# -------------------------
def get_collections_for_platform(platform):
    if platform == "LC08":
        return [
            "LANDSAT/LC08/C02/T1_TOA",
            "LANDSAT/LC08/C02/T2_TOA",
        ]
    elif platform == "LC09":
        return [
            "LANDSAT/LC09/C02/T1_TOA",
            "LANDSAT/LC09/C02/T2_TOA",
        ]
    else:
        raise ValueError(f"Unknown platform: {platform}")

def merge_collections(col_ids):
    ic = ee.ImageCollection(col_ids[0])
    for cid in col_ids[1:]:
        ic = ic.merge(ee.ImageCollection(cid))
    return ic

def parse_scene_key(image_index):
    m = re.match(r"^(LC0[89])_(\d{6})_(\d{8})$", image_index)
    if not m:
        raise ValueError(f"Bad IMAGE_INDEX format: {image_index}")
    platform = m.group(1)
    pathrow = m.group(2)
    yyyymmdd = m.group(3)

    wrs_path = int(pathrow[:3])
    wrs_row = int(pathrow[3:])

    year = int(yyyymmdd[:4])
    month = int(yyyymmdd[4:6])
    day = int(yyyymmdd[6:8])

    return platform, wrs_path, wrs_row, year, month, day, yyyymmdd

def resolve_reference_image(image_index, roi, search_pad_days=2):
    """
    Resolve short key like LC08_017042_20200101 to the best matching GEE image.
    """
    platform, wrs_path, wrs_row, year, month, day, yyyymmdd = parse_scene_key(image_index)

    target_date = ee.Date.fromYMD(year, month, day)
    start = target_date.advance(-search_pad_days, "day")
    end = target_date.advance(search_pad_days + 1, "day")

    ic = (
        merge_collections(get_collections_for_platform(platform))
        .filterDate(start, end)
        .filterBounds(roi)
        .filter(ee.Filter.eq("WRS_PATH", wrs_path))
        .filter(ee.Filter.eq("WRS_ROW", wrs_row))
    )

    n = ic.size().getInfo()
    if n == 0:
        raise RuntimeError(
            f"Could not resolve reference image for {image_index} "
            f"within ±{search_pad_days} days."
        )

    target_ms = target_date.millis()

    def add_dt(img):
        abs_dt = ee.Number(img.get("system:time_start")).subtract(target_ms).abs()
        return img.set("ABS_DT_MS", abs_dt)

    best = ee.Image(ic.map(add_dt).sort("ABS_DT_MS", True).first())
    return best

def clouds_from_qapixel(img):
    qa = img.select("QA_PIXEL")
    dilated = qa.bitwiseAnd(1 << 1).neq(0)
    cirrus  = qa.bitwiseAnd(1 << 2).neq(0)
    cloud   = qa.bitwiseAnd(1 << 3).neq(0)
    shadow  = qa.bitwiseAnd(1 << 4).neq(0)
    m = dilated.Or(cirrus).Or(cloud).Or(shadow)
    if CLOUD_DILATE_M and CLOUD_DILATE_M > 0:
        m = m.focal_max(radius=CLOUD_DILATE_M, units="meters")
    return m.rename("cloud")

# Ocean mask
ocean_mask = multitemporal_cloud_masking_2.GetOceanMask(
    region_of_interest,
    water_occ_threshold=50
)

def attach_cc_ocean(img):
    cloud_ocean = clouds_from_qapixel(img).updateMask(ocean_mask)

    d = ee.Dictionary(cloud_ocean.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region_of_interest,
        scale=CC_SCALE,
        bestEffort=True,
        tileScale=TILESCALE,
        maxPixels=1e13
    ))

    raw_cloud = d.get("cloud")

    # If reduceRegion returns null, assign 100% cloud as a safe penalty
    cf = ee.Number(ee.Algorithms.If(raw_cloud, raw_cloud, 1.0))
    cc = cf.multiply(100.0)

    return img.set({"CC_OCEAN": cc})

# -------------------------
# Build IMAGE_INDEX list from local folders
# -------------------------
scene_dirs = sorted([p for p in SCENES_DIR.iterdir() if p.is_dir()])
print("Total local scene folders found:", len(scene_dirs))

image_indices = []
for scene_dir in scene_dirs:
    name = scene_dir.name
    m = re.match(r"(LC0[89])_L2SP_(\d{6})_(\d{8})_", name)
    if not m:
        continue
    platform = m.group(1)
    pathrow  = m.group(2)
    acqdate  = m.group(3)
    image_indices.append(f"{platform}_{pathrow}_{acqdate}")

if MAX_SCENES_TO_PROCESS is not None:
    image_indices = image_indices[:MAX_SCENES_TO_PROCESS]

if len(image_indices) == 0:
    raise RuntimeError("No local scenes found to process.")

REFERENCE_INDEX = image_indices[0]
print("\nREFERENCE_INDEX =", REFERENCE_INDEX)

# -------------------------
# Resolve reference image safely
# -------------------------
target = resolve_reference_image(
    REFERENCE_INDEX,
    region_of_interest,
    search_pad_days=SEARCH_PAD_DAYS
)

target_id = target.get("system:id").getInfo()
print("Resolved target system:id =", target_id)

target_time = ee.Date(target.get("system:time_start"))
target_ts   = ee.Number(target.get("system:time_start"))

start_time = target_time.advance(-LAG_WINDOW_YEARS, "year")
end_time   = target_time.advance( LAG_WINDOW_YEARS, "year")

wrs_path = ee.Number(target.get("WRS_PATH"))
wrs_row  = ee.Number(target.get("WRS_ROW"))

# -------------------------
# Build pool in ± window, compute CC_OCEAN server-side, sort by CC_OCEAN
# -------------------------
pool = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA")
    .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_TOA"))
    .filterDate(start_time, end_time)
    .filterBounds(region_of_interest)
    .filter(ee.Filter.eq("WRS_PATH", wrs_path))
    .filter(ee.Filter.eq("WRS_ROW",  wrs_row))
)

# Optional: exclude dates too near the target
pool = pool.filter(ee.Filter.Or(
    ee.Filter.lt("system:time_start", target_time.advance(-EXCLUDE_DAYS_AROUND_TARGET, "day").millis()),
    ee.Filter.gt("system:time_start", target_time.advance( EXCLUDE_DAYS_AROUND_TARGET, "day").millis())
))

pool_ranked = pool.map(attach_cc_ocean).sort("CC_OCEAN", True)

n_total = pool_ranked.size().getInfo()
print(f"\nTotal candidate scenes in ±{LAG_WINDOW_YEARS} years:", n_total)

n_print = min(TOP_N_PRINT, n_total)
lst = pool_ranked.toList(n_print).getInfo()
target_ts_py = target_ts.getInfo()

print("\nRank | Date       | Sensor | CC_ocean(%) | Δt(days) | Image ID")
print("-" * 110)

rows = []
for i, feat in enumerate(lst, start=1):
    props = feat["properties"]
    img_id = feat.get("id", "NA")
    cc = props.get("CC_OCEAN", None)
    ts = props.get("system:time_start", None)

    date_str = "NA"
    delta_days = None
    if ts is not None:
        date_str = datetime.utcfromtimestamp(ts / 1000).strftime("%Y-%m-%d")
        delta_days = abs(ts - target_ts_py) / (1000 * 60 * 60 * 24)

    sensor = "LC08" if "/LC08_" in img_id else ("LC09" if "/LC09_" in img_id else "Lx")

    print(f"{i:4d} | {date_str} | {sensor:>5s} | {cc:10.2f} | {delta_days:7.1f} | {img_id}")
    rows.append((float(cc), float(delta_days), int(ts), sensor, img_id, date_str))

# -------------------------
# Select FIXED_LAGS
# -------------------------
good = [r for r in rows if r[0] < CC_OCEAN_MAX_PCT]
good.sort(key=lambda r: (r[0], r[1]))

if len(good) == 0:
    print(f"\n[LAGS] WARNING: no scenes with CC_ocean < {CC_OCEAN_MAX_PCT:.2f}% in top {n_print}.")
    print("[LAGS] Falling back to best-by-(cc,Δt) from printed list.")
    good = rows[:]
    good.sort(key=lambda r: (r[0], r[1]))

FIXED_LAGS = good[:NUM_LAGS]

print("\n[LAGS] FIXED TOP LAGS (used for ALL scenes):")
for k, (cc, delta_days, ts, sensor, pid, dt) in enumerate(FIXED_LAGS, start=1):
    print(f"  Lag {k}: {dt} | {sensor} | CC_ocean={cc:.2f}% | Δt={delta_days:.1f} days | {pid}")

# Optional sanity check
want_id = "LANDSAT/LC08/C02/T1_TOA/LC08_064045_20250123"
print("\nContains 2025-01-23 in TOP_N_PRINT list?", any(r[4] == want_id for r in rows))

Total local scene folders found: 230

REFERENCE_INDEX = LC08_017042_20200101
Resolved target system:id = LANDSAT/LC08/C02/T2_TOA/LC08_017042_20200101

Total candidate scenes in ±7 years: 1

Rank | Date       | Sensor | CC_ocean(%) | Δt(days) | Image ID
--------------------------------------------------------------------------------------------------------------
   1 | 2013-03-23 |  LC08 |     100.00 |  2475.0 | LANDSAT/LC08/C02/T1_TOA/LC08_017042_20130323

[LAGS] WARNING: no scenes with CC_ocean < 1.00% in top 1.
[LAGS] Falling back to best-by-(cc,Δt) from printed list.

[LAGS] FIXED TOP LAGS (used for ALL scenes):
  Lag 1: 2013-03-23 | LC08 | CC_ocean=100.00% | Δt=2475.0 days | LANDSAT/LC08/C02/T1_TOA/LC08_017042_20130323

Contains 2025-01-23 in TOP_N_PRINT list? False


## Use if you prefer to process the whole scene

In [1]:
# =========================
# CELL 1:
# Find + print best FIXED lags using whole Landsat scene footprint
# No POLY needed
# =========================

import ee
from datetime import datetime
from pathlib import Path
import re

from ee_ipl_uv import multitemporal_cloud_masking_2

ee.Initialize()

# -------------------------
# USER SETTINGS
# -------------------------
SCENES_DIR = Path(r"Y:/Mingyue/Timor_part1/landsat_c2_l2_extracted")
MAX_SCENES_TO_PROCESS = None

# Lag selection knobs
NUM_LAGS = 5
LAG_WINDOW_YEARS = 7
CC_OCEAN_MAX_PCT = 1.0
EXCLUDE_DAYS_AROUND_TARGET = 7

# Ranking / printing
TOP_N_PRINT = 400
CC_SCALE = 120
TILESCALE = 4
CLOUD_DILATE_M = 60

SEARCH_PAD_DAYS = 2


# -------------------------
# Helpers
# -------------------------
def get_collections_for_platform(platform):
    if platform == "LC08":
        return [
            "LANDSAT/LC08/C02/T1_TOA",
            "LANDSAT/LC08/C02/T2_TOA",
        ]
    elif platform == "LC09":
        return [
            "LANDSAT/LC09/C02/T1_TOA",
            "LANDSAT/LC09/C02/T2_TOA",
        ]
    else:
        raise ValueError(f"Unknown platform: {platform}")


def merge_collections(col_ids):
    ic = ee.ImageCollection(col_ids[0])
    for cid in col_ids[1:]:
        ic = ic.merge(ee.ImageCollection(cid))
    return ic


def parse_scene_key(image_index):
    m = re.match(r"^(LC0[89])_(\d{6})_(\d{8})$", image_index)
    if not m:
        raise ValueError(f"Bad IMAGE_INDEX format: {image_index}")

    platform = m.group(1)
    pathrow = m.group(2)
    yyyymmdd = m.group(3)

    wrs_path = int(pathrow[:3])
    wrs_row = int(pathrow[3:])

    year = int(yyyymmdd[:4])
    month = int(yyyymmdd[4:6])
    day = int(yyyymmdd[6:8])

    return platform, wrs_path, wrs_row, year, month, day, yyyymmdd


def resolve_reference_image(image_index, search_pad_days=2):
    """
    Resolve short key like LC08_017042_20200101 to the best matching GEE image.
    No AOI/POLY is used.
    """
    platform, wrs_path, wrs_row, year, month, day, yyyymmdd = parse_scene_key(image_index)

    target_date = ee.Date.fromYMD(year, month, day)
    start = target_date.advance(-search_pad_days, "day")
    end = target_date.advance(search_pad_days + 1, "day")

    ic = (
        merge_collections(get_collections_for_platform(platform))
        .filterDate(start, end)
        .filter(ee.Filter.eq("WRS_PATH", wrs_path))
        .filter(ee.Filter.eq("WRS_ROW", wrs_row))
    )

    n = ic.size().getInfo()
    if n == 0:
        raise RuntimeError(
            f"Could not resolve reference image for {image_index} "
            f"within ±{search_pad_days} days."
        )

    target_ms = target_date.millis()

    def add_dt(img):
        abs_dt = ee.Number(img.get("system:time_start")).subtract(target_ms).abs()
        return img.set("ABS_DT_MS", abs_dt)

    best = ee.Image(ic.map(add_dt).sort("ABS_DT_MS", True).first())
    return best


def clouds_from_qapixel(img):
    qa = img.select("QA_PIXEL")

    dilated = qa.bitwiseAnd(1 << 1).neq(0)
    cirrus  = qa.bitwiseAnd(1 << 2).neq(0)
    cloud   = qa.bitwiseAnd(1 << 3).neq(0)
    shadow  = qa.bitwiseAnd(1 << 4).neq(0)

    m = dilated.Or(cirrus).Or(cloud).Or(shadow)

    if CLOUD_DILATE_M and CLOUD_DILATE_M > 0:
        m = m.focal_max(radius=CLOUD_DILATE_M, units="meters")

    return m.rename("cloud")


# -------------------------
# Build IMAGE_INDEX list from local folders
# -------------------------
scene_dirs = sorted([p for p in SCENES_DIR.iterdir() if p.is_dir()])
print("Total local scene folders found:", len(scene_dirs))

image_indices = []

for scene_dir in scene_dirs:
    name = scene_dir.name
    m = re.match(r"(LC0[89])_L2SP_(\d{6})_(\d{8})_", name)

    if not m:
        continue

    platform = m.group(1)
    pathrow = m.group(2)
    acqdate = m.group(3)

    image_indices.append(f"{platform}_{pathrow}_{acqdate}")

if MAX_SCENES_TO_PROCESS is not None:
    image_indices = image_indices[:MAX_SCENES_TO_PROCESS]

if len(image_indices) == 0:
    raise RuntimeError("No local scenes found to process.")

REFERENCE_INDEX = image_indices[0]
print("\nREFERENCE_INDEX =", REFERENCE_INDEX)


# -------------------------
# Resolve reference image
# -------------------------
target = resolve_reference_image(
    REFERENCE_INDEX,
    search_pad_days=SEARCH_PAD_DAYS
)

target_id = target.get("system:id").getInfo()
print("Resolved target system:id =", target_id)

# =========================================================
# IMPORTANT CHANGE:
# Use whole Landsat scene footprint instead of POLY
# =========================================================
region_of_interest = target.geometry()

print("\nUsing whole target scene footprint as region_of_interest.")


# -------------------------
# Ocean mask based on whole scene footprint
# -------------------------
ocean_mask = multitemporal_cloud_masking_2.GetOceanMask(
    region_of_interest,
    water_occ_threshold=50
)


def attach_cc_ocean(img):
    cloud_ocean = clouds_from_qapixel(img).updateMask(ocean_mask)

    d = ee.Dictionary(
        cloud_ocean.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region_of_interest,
            scale=CC_SCALE,
            bestEffort=True,
            tileScale=TILESCALE,
            maxPixels=1e13,
        )
    )

    raw_cloud = d.get("cloud")

    # If reduceRegion returns null, assign 100% cloud as safe penalty
    cf = ee.Number(ee.Algorithms.If(raw_cloud, raw_cloud, 1.0))
    cc = cf.multiply(100.0)

    return img.set({"CC_OCEAN": cc})


# -------------------------
# Build pool around target scene
# -------------------------
target_time = ee.Date(target.get("system:time_start"))
target_ts = ee.Number(target.get("system:time_start"))

start_time = target_time.advance(-LAG_WINDOW_YEARS, "year")
end_time = target_time.advance(LAG_WINDOW_YEARS, "year")

wrs_path = ee.Number(target.get("WRS_PATH"))
wrs_row = ee.Number(target.get("WRS_ROW"))

pool = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA")
    .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_TOA"))
    .filterDate(start_time, end_time)
    .filter(ee.Filter.eq("WRS_PATH", wrs_path))
    .filter(ee.Filter.eq("WRS_ROW", wrs_row))
)

# Optional: exclude dates too near the target
pool = pool.filter(
    ee.Filter.Or(
        ee.Filter.lt(
            "system:time_start",
            target_time.advance(-EXCLUDE_DAYS_AROUND_TARGET, "day").millis(),
        ),
        ee.Filter.gt(
            "system:time_start",
            target_time.advance(EXCLUDE_DAYS_AROUND_TARGET, "day").millis(),
        ),
    )
)

pool_ranked = pool.map(attach_cc_ocean).sort("CC_OCEAN", True)

n_total = pool_ranked.size().getInfo()
print(f"\nTotal candidate scenes in ±{LAG_WINDOW_YEARS} years:", n_total)

n_print = min(TOP_N_PRINT, n_total)
lst = pool_ranked.toList(n_print).getInfo()
target_ts_py = target_ts.getInfo()

print("\nRank | Date       | Sensor | CC_ocean(%) | Δt(days) | Image ID")
print("-" * 110)

rows = []

for i, feat in enumerate(lst, start=1):
    props = feat["properties"]
    img_id = feat.get("id", "NA")
    cc = props.get("CC_OCEAN", None)
    ts = props.get("system:time_start", None)

    date_str = "NA"
    delta_days = None

    if ts is not None:
        date_str = datetime.utcfromtimestamp(ts / 1000).strftime("%Y-%m-%d")
        delta_days = abs(ts - target_ts_py) / (1000 * 60 * 60 * 24)

    sensor = "LC08" if "/LC08_" in img_id else ("LC09" if "/LC09_" in img_id else "Lx")

    print(
        f"{i:4d} | {date_str} | {sensor:>5s} | "
        f"{cc:10.2f} | {delta_days:7.1f} | {img_id}"
    )

    rows.append((float(cc), float(delta_days), int(ts), sensor, img_id, date_str))


# -------------------------
# Select FIXED_LAGS
# -------------------------
good = [r for r in rows if r[0] < CC_OCEAN_MAX_PCT]
good.sort(key=lambda r: (r[0], r[1]))

if len(good) == 0:
    print(
        f"\n[LAGS] WARNING: no scenes with CC_ocean < "
        f"{CC_OCEAN_MAX_PCT:.2f}% in top {n_print}."
    )
    print("[LAGS] Falling back to best-by-(cc, Δt) from printed list.")

    good = rows[:]
    good.sort(key=lambda r: (r[0], r[1]))

FIXED_LAGS = good[:NUM_LAGS]

print("\n[LAGS] FIXED TOP LAGS (used for ALL scenes):")

for k, (cc, delta_days, ts, sensor, pid, dt) in enumerate(FIXED_LAGS, start=1):
    print(
        f"  Lag {k}: {dt} | {sensor} | "
        f"CC_ocean={cc:.2f}% | Δt={delta_days:.1f} days | {pid}"
    )

Total local scene folders found: 232

REFERENCE_INDEX = LC08_110066_20200113
Resolved target system:id = LANDSAT/LC08/C02/T1_TOA/LC08_110066_20200113

Using whole target scene footprint as region_of_interest.

Total candidate scenes in ±7 years: 379

Rank | Date       | Sensor | CC_ocean(%) | Δt(days) | Image ID
--------------------------------------------------------------------------------------------------------------
   1 | 2015-09-28 |  LC08 |       0.01 |  1568.0 | LANDSAT/LC08/C02/T1_TOA/LC08_110066_20150928
   2 | 2018-08-19 |  LC08 |       0.01 |   512.0 | LANDSAT/LC08/C02/T1_TOA/LC08_110066_20180819
   3 | 2015-09-12 |  LC08 |       0.02 |  1584.0 | LANDSAT/LC08/C02/T1_TOA/LC08_110066_20150912
   4 | 2014-09-09 |  LC08 |       0.04 |  1952.0 | LANDSAT/LC08/C02/T1_TOA/LC08_110066_20140909
   5 | 2016-07-12 |  LC08 |       0.07 |  1280.0 | LANDSAT/LC08/C02/T1_TOA/LC08_110066_20160712
   6 | 2019-09-07 |  LC08 |       0.09 |   128.0 | LANDSAT/LC08/C02/T1_TOA/LC08_110066_20190907

In [ ]:
# =========================
# CELL 2: Batch export ALL scenes (223) with robust resolver + debug counters
# =========================

import ee
import time
from datetime import datetime
from tqdm.auto import tqdm

from ee_ipl_uv import normalization, clustering

ee.Initialize()

# -------------------------
# BATCH / THROTTLE SETTINGS
# -------------------------
BATCH_SIZE = 25                # submit 25 tasks per run
SLEEP_BETWEEN_TASKS_S = 1.5    # safer; 0.35 is often too fast for 200+ tasks
PAUSE_BETWEEN_BATCHES_S = 10   # short pause after each batch submission

# -------------------------
# RESOLVER SETTINGS
# -------------------------
SEARCH_PAD_DAYS = 2            # fallback search: date ±2 days
MAX_CANDIDATES = 30            # safety cap

# -------------------------
# MASK SETTINGS (same as yours)
# -------------------------
N_CLUSTERS = 10
NUM_PIXELS = 600

THRESHOLD_DIFFERENCE  = 0.04
THRESHOLD_REFLECTANCE = 0.175
P_YELLOW              = 98
SOFT_REFL_FACTOR      = 0.80

THRESHOLD_SHADOW_REFLECTANCE = 0.16
THRESHOLD_SHADOW_DIFF        = 0.03

EXPORT_SCALE      = 30
EXPORT_FOLDER     = "Timor_110_066_L89_cloudshadow_masks"
EXPORT_MAX_PIXELS = 1e13

# -------------------------
# Water mask
# -------------------------
def get_water_mask_mod44w(scene_bounds_geom):
    mod44w = ee.ImageCollection("MODIS/006/MOD44W").sort("system:time_start", False).first()
    wm = mod44w.select("water_mask")
    water01 = wm.bitwiseAnd(1).eq(1).unmask(0).toByte()
    return water01.clip(scene_bounds_geom)

# -------------------------
# Build image_with_lags from FIXED_LAGS
# -------------------------
REFL_BANDS = ["B1","B2","B3","B4","B5","B6","B7","B8","B9","B10","B11"]

def build_image_with_fixed_lags(target_img, fixed_lags):
    img_with = target_img.select(REFL_BANDS).set("n_lags_found", len(fixed_lags))
    for i, (cc, delta_days, ts, sensor, pid, dt) in enumerate(fixed_lags, start=1):
        lag_img = ee.Image(pid)
        new_names = [b + f"_lag_{i}" for b in REFL_BANDS]
        img_with = (img_with
            .addBands(lag_img.select(REFL_BANDS).rename(new_names))
            .set(f"CC_OCEAN_lag_{i}", float(cc))
            .set(f"system:time_start_lag_{i}", int(ts))
            .set(f"system:id_lag_{i}", pid)
        )
    return img_with

# -------------------------
# Fast: try direct ID in T1_TOA (your current method)
# -------------------------
def try_direct_image(platform, image_index):
    col = "LANDSAT/LC08/C02/T1_TOA" if platform == "LC08" else "LANDSAT/LC09/C02/T1_TOA"
    img = ee.Image(f"{col}/{image_index}")
    # If it doesn't exist, this getInfo will throw
    _ = img.get("system:time_start").getInfo()
    return img, col

# -------------------------
# Fallback: resolve using date+WRS+ROI in T1_TOA + T2_TOA (±SEARCH_PAD_DAYS)
# -------------------------
def resolve_toa_fallback(platform, pathrow, yyyymmdd, roi):
    wrs_path = int(pathrow[:3])
    wrs_row  = int(pathrow[3:])

    y = int(yyyymmdd[0:4]); m = int(yyyymmdd[4:6]); d = int(yyyymmdd[6:8])
    day0 = ee.Date.fromYMD(y, m, d)
    start = day0.advance(-SEARCH_PAD_DAYS, "day")
    end   = day0.advance(SEARCH_PAD_DAYS + 1, "day")

    if platform == "LC08":
        cols = ["LANDSAT/LC08/C02/T1_TOA", "LANDSAT/LC08/C02/T2_TOA"]
    else:
        cols = ["LANDSAT/LC09/C02/T1_TOA", "LANDSAT/LC09/C02/T2_TOA"]

    ic = ee.ImageCollection(cols[0])
    for c in cols[1:]:
        ic = ic.merge(ee.ImageCollection(c))

    ic = (ic.filterDate(start, end)
            .filterBounds(roi)
            .filter(ee.Filter.eq("WRS_PATH", wrs_path))
            .filter(ee.Filter.eq("WRS_ROW",  wrs_row))
            .sort("system:time_start", True)
         )

    n = ic.size().getInfo()
    if n == 0:
        return None, cols, n

    # choose closest to day0
    target_ms = day0.millis()
    def add_dt(img):
        dt = ee.Number(img.get("system:time_start")).subtract(target_ms).abs()
        return img.set("ABS_DT_MS", dt)

    best = ee.Image(ic.map(add_dt).sort("ABS_DT_MS", True).first())
    return best, cols, n

# -------------------------
# Counters / logs
# -------------------------
n8 = sum(idx.startswith("LC08_") for idx in image_indices)
n9 = sum(idx.startswith("LC09_") for idx in image_indices)
print(f"Targets loaded: LC08={n8}, LC09={n9}, total={len(image_indices)}")

submitted_8 = submitted_9 = 0
fallback_used_8 = fallback_used_9 = 0
not_found = []   # (IMAGE_INDEX, reason)
failed = []      # (IMAGE_INDEX, reason)

# -------------------------
# Submit in batches
# -------------------------
total = len(image_indices)
batches = [image_indices[i:i+BATCH_SIZE] for i in range(0, total, BATCH_SIZE)]

print(f"Submitting {total} scenes in {len(batches)} batches of {BATCH_SIZE}...")

for bi, batch in enumerate(batches, start=1):
    print(f"\n=== BATCH {bi}/{len(batches)}: {len(batch)} scenes ===")

    pbar = tqdm(batch, desc=f"Batch {bi} submit", unit="scene")

    for IMAGE_INDEX in pbar:
        try:
            platform, pathrow, yyyymmdd = IMAGE_INDEX.split("_", 2)

            # 1) direct
            img = None
            try:
                img, col = try_direct_image(platform, IMAGE_INDEX)
                used_fallback = False
            except Exception:
                # 2) fallback
                img, cols, n = resolve_toa_fallback(platform, pathrow, yyyymmdd, region_of_interest)
                used_fallback = True

                if img is None:
                    not_found.append((IMAGE_INDEX, f"No TOA in T1/T2 within ±{SEARCH_PAD_DAYS} days"))
                    pbar.set_postfix({"sub8": submitted_8, "sub9": submitted_9, "nf": len(not_found), "fail": len(failed)})
                    continue

            # Geometry / ROI
            scene_geom   = img.geometry()
            ROI_EFF      = region_of_interest.intersection(scene_geom, ee.ErrorMargin(1))
            scene_bounds = scene_geom.bounds()

            # Water mask
            water_mask_scene = get_water_mask_mod44w(scene_bounds)
            water_mask_roi   = water_mask_scene.clip(ROI_EFF)

            def waterize(x):
                return x.updateMask(water_mask_roi).clip(ROI_EFF)

            # Add fixed lags
            image_with_lags = build_image_with_fixed_lags(img, FIXED_LAGS)

            # ---- cloud mask2 ----
            reflectance_bands = REFL_BANDS[:]
            reflectance_lag1  = [b + "_lag_1" for b in reflectance_bands]

            image_to_predict      = waterize(image_with_lags.select(reflectance_bands))
            background_prediction = waterize(image_with_lags.select(reflectance_lag1))

            img_differences_pos = image_to_predict.subtract(background_prediction).max(0)

            training = img_differences_pos.sample(
                region=ROI_EFF,
                scale=30,
                numPixels=NUM_PIXELS,
                dropNulls=True,
                geometries=False
            )

            training, media, std = normalization.ComputeNormalizationFeatureCollection(training, reflectance_bands)
            clusterer = ee.Clusterer.wekaKMeans(N_CLUSTERS).train(training)

            img_differences_norm = normalization.ApplyNormalizationImage(img_differences_pos, reflectance_bands, media, std)
            result = waterize(img_differences_norm.cluster(clusterer)).rename("cluster_id")

            multitemporal_cloud_score, reflectance_score = clustering.SelectClusters(
                image_to_predict,
                background_prediction,
                result,
                n_clusters=N_CLUSTERS,
                region_of_interest=ROI_EFF
            )

            score = waterize(multitemporal_cloud_score).rename("score")
            refl  = waterize(reflectance_score).rename("refl")
            score_norm = score.divide(ee.Number(THRESHOLD_DIFFERENCE)).clamp(0, 1).rename("score_norm")

            pct = ee.Dictionary(score_norm.reduceRegion(
                reducer=ee.Reducer.percentile([P_YELLOW]),
                geometry=ROI_EFF,
                scale=120,
                bestEffort=True,
                tileScale=4,
                maxPixels=1e13
            ))

            key = f"score_norm_p{P_YELLOW}"
            thr_yellow = ee.Number(ee.Algorithms.If(pct.contains(key), pct.get(key), 0.97))

            cloud_mask2 = score_norm.gte(thr_yellow).rename("cloud").uint8().updateMask(water_mask_roi)
            cloud_mask2 = cloud_mask2.And(
                refl.gt(ee.Number(THRESHOLD_REFLECTANCE).multiply(SOFT_REFL_FACTOR))
            ).rename("cloud").uint8().updateMask(water_mask_roi)
            cloud_mask2 = cloud_mask2.focal_max(radius=30, units="meters")

            # ---- shadow mask ----
            visnir_now  = ["B4", "B3", "B2", "B5"]
            visnir_lag1 = [b + "_lag_1" for b in visnir_now]

            mean_now_4 = image_with_lags.select(visnir_now).reduce(ee.Reducer.mean())
            mean_lag_4 = image_with_lags.select(visnir_lag1).reduce(ee.Reducer.mean())
            diff_4     = mean_now_4.subtract(mean_lag_4)

            shadow_score = diff_4.multiply(-1).max(0).rename("shadow_score")

            vis_mean_now = image_with_lags.select(["B4","B3","B2"]).reduce(ee.Reducer.mean())
            dark_mask    = vis_mean_now.lt(THRESHOLD_SHADOW_REFLECTANCE).rename("dark_mask")

            shadow_score = waterize(shadow_score).updateMask(waterize(dark_mask))
            shadow_mask  = shadow_score.gt(THRESHOLD_SHADOW_DIFF).rename("shadow").uint8().updateMask(water_mask_roi)
            shadow_mask  = shadow_mask.focal_max(radius=30, units="meters")

            # ---- combined 3-class ----
            cloud_u  = cloud_mask2.unmask(0).uint8()
            shadow_u = shadow_mask.unmask(0).uint8()
            cloud_or_shadow = cloud_u.Or(shadow_u).rename("cloud_or_shadow").uint8()

            combined = ee.Image(0).rename("mask").uint8()
            combined = combined.where(cloud_or_shadow.eq(1), 1)
            combined = combined.where(water_mask_roi.eq(0), 2)
            combined = combined.clip(ROI_EFF).unmask(0).toByte()

            # Export
            export_name = f"{IMAGE_INDEX}_mask_cloudshadow_land3class_fixedlags"
            task = ee.batch.Export.image.toDrive(
                image=combined,
                description=export_name,
                folder=EXPORT_FOLDER,
                fileNamePrefix=export_name,
                region=ROI_EFF,
                scale=EXPORT_SCALE,
                maxPixels=EXPORT_MAX_PIXELS
            )
            task.start()

            # Counters
            if platform == "LC08":
                submitted_8 += 1
                if used_fallback:
                    fallback_used_8 += 1
            else:
                submitted_9 += 1
                if used_fallback:
                    fallback_used_9 += 1

            pbar.set_postfix({
                "sub8": submitted_8, "sub9": submitted_9,
                "fb8": fallback_used_8, "fb9": fallback_used_9,
                "nf": len(not_found), "fail": len(failed)
            })

            time.sleep(SLEEP_BETWEEN_TASKS_S)

        except Exception as e:
            failed.append((IMAGE_INDEX, str(e)))
            pbar.set_postfix({
                "sub8": submitted_8, "sub9": submitted_9,
                "fb8": fallback_used_8, "fb9": fallback_used_9,
                "nf": len(not_found), "fail": len(failed)
            })
            continue

    # short pause between batches
    time.sleep(PAUSE_BETWEEN_BATCHES_S)

print("\n================ SUMMARY ================")
print("SUBMITTED total:", submitted_8 + submitted_9, f"(LC08={submitted_8}, LC09={submitted_9})")
print("Fallback used:", fallback_used_8 + fallback_used_9, f"(LC08={fallback_used_8}, LC09={fallback_used_9})")
print("NOT FOUND:", len(not_found))
print("FAILED (other errors):", len(failed))

if not_found:
    print("\nFirst 20 NOT FOUND:")
    for idx, reason in not_found[:20]:
        print(" -", idx, "|", reason)

if failed:
    print("\nFirst 20 FAILED:")
    for idx, reason in failed[:20]:
        print(" -", idx, "|", reason)

print("\nDrive export folder:", EXPORT_FOLDER)
print("IMPORTANT: Open Earth Engine Tasks tab and monitor progress. Submit next run only after tasks complete.")


Targets loaded: LC08=137, LC09=95, total=232
Submitting 232 scenes in 10 batches of 25...

=== BATCH 1/10: 25 scenes ===


Batch 1 submit:   0%|          | 0/25 [00:00<?, ?scene/s]